# Positioning the IQM-Based Mixed-Effects Harmonisation Framework

In this course, we will demonstrate an IQM-based harmonisation workflow using imaging-derived phenotypes (IDPs) from the [ON-Harmony](https://openneuro.org/datasets/ds004712/versions/2.0.1) dataset.

The framework implements a relatively lightweight and interpretable harmonisation strategy within a linear mixed-effects modelling framework. Rather than modelling scanner or site effects directly at the level of the imaging-derived phenotypes (IDPs), the approach uses image quality metrics (IQMs) to identify latent nuisance structure associated with scanner and acquisition variability. Additive and multiplicative correction is then applied selectively using mixed-effects regression models.

Conceptually, the framework can be viewed as a QC-informed harmonisation approach in which:

- scanner/site information is used to identify nuisance-related QC structure,
- biologically meaningful covariates are explicitly protected during QC selection,
- and correction is applied only when statistically justified.

Importantly, harmonisation is not imposed uniformly across all observations or phenotypes. Instead, the framework adaptively determines:

- which QC components are relevant for a given IDP,
- whether additive correction is appropriate,
- and whether multiplicative scaling sufficiently improves model fit to justify correction.

This differs from direct batch-normalisation approaches in which scanner/site correction is applied explicitly at the outcome level.

The framework is intentionally simple and transparent:

- modelling assumptions are linear,
- nuisance structure remains interpretable through QC variables,
- and harmonisation can be evaluated using standard mixed-effects diagnostics.

Such an approach may be particularly useful in:

- smaller clinical datasets,
- longitudinal studies,
- or settings with limited numbers of subjects per scanner/site, where highly flexible machine-learning approaches may be difficult to estimate robustly.

At the same time, this workflow represents only one point along a broader spectrum of IQM-based harmonisation strategies.

More advanced approaches, such as BARTharm, extend QC-informed harmonisation into substantially more flexible modelling frameworks. BARTharm uses Bayesian Additive Regression Trees (BART) to capture nonlinear relationships and higher-order interactions between image-quality structure, scanner effects, and biological variation. These approaches may provide a more robust representation of complex nuisance structure in large and heterogeneous datasets where scanner effects are unlikely to be adequately described by linear relationships alone.

Accordingly, the present workflow should be viewed as:

- a transparent and interpretable mixed-effects implementation,
- a useful pedagogical introduction to IQM-guided harmonisation,
- and a practical approach for smaller or clinically constrained datasets.

Together, these methods illustrate a broader shift away from direct batch-label correction toward harmonisation strategies that explicitly leverage image-quality information to characterize nuisance variation.

In [ ]:
# install packages if required
#%pip install numpy scipy pandas scikit-learn statsmodels matplotlib seaborn

## Inputs Required for the Code

The IDPs and IQMs available from ON-Harmony dataset are available on [OpenNeuro](https://openneuro.org/datasets/ds004712/versions/2.0.1). 

- **onharmony.csv** — contains all IQMs and IDPs for each observation in the dataset. In this dataset we have one IDP to be harmonised and several QC metrics.

- **idp_list.csv** — contains the list of IDPs to be harmonised. These must match the corresponding column names in *simulated_data.csv*.

- **iqm_list.csv** — contains the list of IQMs to be used in the harmonisation workflow. These must match the corresponding column names in *simulated_data.csv*.

## Arguments to the Harmonisation Function

We will use the `iqm_harmonise` function from `iqm_harmonisation_light.py`. We recommend reviewing both the implementation and the inline documentation describing the function arguments before applying the workflow to your own datasets.

## Outputs

After running the IQM-based harmonisation workflow, several output files will be generated:

- *iqm_harmonised.csv*: contains the original and harmonised IDPs (`harmonised_<idp_name>`)
- *qc_selection_<additive/multiplicative>_details.csv*: contains statistical significance measures and summary information for QC selection
- *selected_<additive/multiplicative>_by_volume.csv*: contains the QC variables selected for each IDP

## Visualisation

Next, we will visualise the distribution of IDPs across batches using the outputs from *iqm_harmonised.csv*.

## Conceptual Notes and Limitations

The current implementation still makes use of an explicit batch/scanner variable during the IQM selection stage. In this sense, the workflow remains partially batch-guided, similar to methods such as ComBat, which also rely on known batch labels during harmonisation.

However, unlike direct batch-correction methods, the scanner variable is not used directly to correct the imaging-derived phenotypes (IDPs). Instead, the framework first identifies image quality metrics (IQMs) that appear to capture scanner- or acquisition-related nuisance variation while avoiding IQMs that strongly encode biological or protected covariates. The selected IQMs are then used to guide harmonisation of the IDPs.

Conceptually, this makes the approach more indirect and relatively conservative compared with methods that directly regress out batch effects from all features. Correction is only driven by QC variables that satisfy the predefined statistical selection criteria, rather than by the scanner labels themselves.

One motivation for this strategy is that direct batch-correction approaches can sometimes increase the risk of over-correction, particularly in smaller or clinically heterogeneous datasets where scanner effects may be partially entangled with genuine biological variation. By constraining harmonisation through selected IQMs, the framework attempts to reduce removal of biologically meaningful signal while still mitigating nuisance variation.

More advanced IQM-driven approaches, such as BARTharm, move further away from explicit batch-guided correction by learning nuisance structure directly from image-quality patterns using more flexible nonlinear modelling strategies.

Regardless of the harmonisation method used, harmonised outputs should always be evaluated carefully using multiple complementary metrics and visualisations to assess both nuisance removal and preservation of biological signal.

In [ ]:
# apply iqm harmonisation code
from scipy.stats import zscore
import pandas as pd
import importlib
import iqm_harmonisation_light

importlib.reload(iqm_harmonisation_light)

from iqm_harmonisation_light import iqm_harmonise

df = pd.read_csv("onharmony.csv")
idp_list = pd.read_csv("IDP_list.csv", header=None).iloc[:, 0].dropna().astype(str).str.strip().tolist()
iqm_list = pd.read_csv("IQM_list.csv", header=None).iloc[:, 0].dropna().astype(str).str.strip().tolist()


data_out, qc_selection, add_detail_df, mult_detail_df, summary_df = iqm_harmonise(
    data=df,
    idp_list=idp_list,
    qc_list=iqm_list,
    preserve_covars=("age",),
    adjust_covars=("timepoint",),
    reverse_guard_covars=("age",),
    categorical_covars=("timepoint", "scan_session"),
    batch_col="scan_session",
    subject_col="subject",
    age_source_col="age",
    batch_source_col="scan_session",
    age_col="age",
    iqm_variance=95,
    p_thr=0.05,
    apply_pca=True,
    verbose_model_fits=True,
    enable_multiplicative=True
)